<a href="https://colab.research.google.com/github/TJOETJOE/uts-nlp/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import glob
import re

In [ ]:
# load dataset

csv_files = glob.glob("/content/*.csv")

all_data = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

print("Total rows:", len(all_data))
all_data.head()


Total rows: 1500


,publishedAt,authorDisplayName,textDisplay,likeCount
0,2025-11-25T16:39:26Z,@jimicicchini,This shows trump firstly was always a democrat...,0
1,2025-11-25T16:36:40Z,@brandoncarbaugh7994,It seems pretty clear that Mamdani will work w...,0
2,2025-11-25T16:36:32Z,@clarencegood1705,"Thos dude is a broken record , same thing over...",0
3,2025-11-25T16:34:08Z,@panjamutimosahary926,"So funny, he said he is the greatest mayor of ...",0
4,2025-11-25T16:36:00Z,@EmbracedsIL,Almost like people who say they have a very la...,0


In [ ]:
all_data=all_data.dropna(subset=['textDisplay'])


def cleaning(text):
    text = text.lower()                                  # lowercase
    text = re.sub(r"http\S+", " ", text)                 # remove links
    text = re.sub(r"@\w+", " ", text)                    # remove @username
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)          # remove symbols & emoji
    text = re.sub(r"\s+", " ", text).strip()             # remove extra spaces
    return text

all_data['clean_text'] = all_data['textDisplay'].apply(cleaning)

all_data.head()


,publishedAt,authorDisplayName,textDisplay,likeCount,clean_text
0,2025-11-25T16:39:26Z,@jimicicchini,This shows trump firstly was always a democrat...,0,this shows trump firstly was always a democrat...
1,2025-11-25T16:36:40Z,@brandoncarbaugh7994,It seems pretty clear that Mamdani will work w...,0,it seems pretty clear that mamdani will work w...
2,2025-11-25T16:36:32Z,@clarencegood1705,"Thos dude is a broken record , same thing over...",0,thos dude is a broken record same thing over a...
3,2025-11-25T16:34:08Z,@panjamutimosahary926,"So funny, he said he is the greatest mayor of ...",0,so funny he said he is the greatest mayor of n...
4,2025-11-25T16:36:00Z,@EmbracedsIL,Almost like people who say they have a very la...,0,almost like people who say they have a very la...


In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]           # remove stopwords
    tokens = [lemmatizer.lemmatize(t) for t in tokens]            # lemmatize
    return " ".join(tokens)

all_data['prep_text'] = all_data['clean_text'].apply(preprocess)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
from textblob import TextBlob

def get_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0:
        return "positive"
    elif polarity < 0:
        return "negative"
    else:
        return "neutral"

all_data['sentiment'] = all_data['prep_text'].apply(get_sentiment)


In [ ]:
all_data['sentiment'].value_counts()


,count
sentiment,
positive,646
neutral,521
negative,333


In [ ]:
print(all_data.columns)
print(all_data['sentiment'].value_counts())
print("Total rows:", len(all_data))

Index(['publishedAt', 'authorDisplayName', 'textDisplay', 'likeCount',
       'clean_text', 'prep_text', 'sentiment'],
      dtype='object')
sentiment
positive    646
neutral     521
negative    333
Name: count, dtype: int64
Total rows: 1500


In [ ]:
# split data
from sklearn.model_selection import train_test_split

X = all_data['prep_text']        # fitur teks
y = all_data['sentiment']        # label sentimen

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,               # 80% train, 20% test
    random_state=42,
    stratify=y                   # supaya proporsi label seimbang
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))


Train size: 1200
Test size : 300


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1,2),      # unigram + bigram
    min_df=2,               # kata muncul minimal di 2 dokumen
    max_df=0.95             # buang kata yang terlalu sering muncul
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

X_train_tfidf.shape, X_test_tfidf.shape


((1200, 2665), (300, 2665))

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)

print("=== Naive Bayes Performance ===")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))


=== Naive Bayes Performance ===
Accuracy: 0.53

Classification Report:
               precision    recall  f1-score   support

    negative       1.00      0.10      0.19        67
     neutral       0.63      0.31      0.41       104
    positive       0.50      0.93      0.65       129

    accuracy                           0.53       300
   macro avg       0.71      0.45      0.42       300
weighted avg       0.65      0.53      0.46       300


Confusion Matrix:
 [[  7  10  50]
 [  0  32  72]
 [  0   9 120]]


In [ ]:
from sklearn.linear_model import LogisticRegression

logreg_model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)
logreg_model.fit(X_train_tfidf, y_train)

y_pred_lr = logreg_model.predict(X_test_tfidf)

print("=== Logistic Regression Performance ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))


=== Logistic Regression Performance ===
Accuracy: 0.62

Classification Report:
               precision    recall  f1-score   support

    negative       0.81      0.25      0.39        67
     neutral       0.58      0.64      0.61       104
    positive       0.62      0.79      0.70       129

    accuracy                           0.62       300
   macro avg       0.67      0.56      0.56       300
weighted avg       0.65      0.62      0.60       300


Confusion Matrix:
 [[ 17  25  25]
 [  0  67  37]
 [  4  23 102]]


In [ ]:
# gunakan seluruh data
X_all_tfidf = tfidf.transform(all_data['prep_text'])

all_data['pred_sentiment'] = nb_model.predict(X_all_tfidf)

print(all_data['pred_sentiment'].value_counts())


pred_sentiment
positive    957
neutral     407
negative    136
Name: count, dtype: int64
